# Fundamentals of Data Science  
## Week 5: Data Preparation II (Feature Engineering)

---



## Table of Contents
1. Learning Objectives
2. Context & Motivation
3. Conceptual Foundation
4. Datasets
5. Time-based Feature Engineering
6. Aggregation-based Feature Enginerring
7. Interaction & Ratio Feature Enginerring
8. Distribution Transform
9. Domain-driven Feature Engineering
10. Statistical Feature Engineering
11. Hands-on: PM2.5 Dataset
12. Deliverable Checklist

---

# 1) Learning Objectives

เมื่อจบคาบนี้ นักศึกษาจะสามารถ

1. อธิบายแนวคิดและบทบาทของ Feature Engineering ต่อคุณภาพโมเดลได้
2. ออกแบบและสร้าง feature ใหม่จากข้อมูลเชิงเวลา เชิงโดเมน และเชิงสถิติได้
3. ประเมินความเหมาะสมและความเสี่ยง (bias / leakage) ของ feature ที่สร้างขึ้น

---

# 2) Context & Motivation

**ทำไปเพื่ออะไร**

* ข้อมูลที่สะอาด ≠ ข้อมูลที่โมเดลเข้าใจ
* โมเดลเรียนรู้จาก *representation* ไม่ใช่ข้อมูลดิบ

**ถ้าไม่ทำจะเกิดอะไรขึ้น**

* โมเดลเรียนรู้ช้า / ไม่เสถียร / อธิบายไม่ได้
* ผลลัพธ์อาจสะท้อน noise มากกว่าสัญญาณจริง

---

# 3) Conceptual Foundation

## 3.1 Feature Engineering คืออะไร

* กระบวนการ **แปลง/สังเคราะห์ตัวแปร** จากข้อมูลดิบให้สะท้อนพฤติกรรมหรือกลไกที่ซ่อนอยู่
* ไม่ใช่ขั้นตอนอัตโนมัติ ต้องอาศัย **บริบทปัญหา + โดเมน**

![Image](https://media.geeksforgeeks.org/wp-content/uploads/20250701114435618562/feature-engineering.webp)

![Image](https://dotdata.com/wp-content/uploads/2022/11/1482323_BlogPostGraphic1_1_101822.png)

![Image](https://learn.microsoft.com/en-us/azure/cloud-adoption-framework/scenarios/cloud-scale-analytics/images/domain-driven-feature-eng-feature-mesh-strategy.png)

## 3.2 ประเภทของ Feature

| ประเภท        | ตัวอย่าง                      | ใช้เมื่อ                |
| ------------- | ----------------------------- | ----------------------- |
| Time-based    | day, month, lag, rolling mean | ข้อมูลมีเวลา/ฤดูกาล     |
| Aggregation   | mean, max, count (ตามกลุ่ม)   | ข้อมูล transactional    |
| Interaction   | x*y, ratio                    | ความสัมพันธ์เชิงผสม     |
| Transform    | log, power transform  | การกระจาย skew / ลดอิทธิพล outlier |
| Domain-driven | กฎจากผู้เชี่ยวชาญ             | มีความรู้เฉพาะโดเมน     |
| Statistical   | z-score, trend                | วิเคราะห์การเปลี่ยนแปลง |

> Feature ที่ดี = *เข้าใจง่าย + อธิบายได้ + ใช้ซ้ำได้*  
> **ข้อควรระวัง:** target leakage, multicollinearity, over-engineering  
---

# 4) Datasets

## 4.1 **Dataset A**: Bike Sharing Demand (Time-based Feature Engineering)

[https://www.kaggle.com/competitions/bike-sharing-demand/data](https://www.kaggle.com/competitions/bike-sharing-demand/data)

### ลักษณะของข้อมูล

* ข้อมูลการเช่าจักรยานราย **ชั่วโมง**
* มีตัวแปรด้าน

  * เวลา (datetime)
  * สภาพอากาศ (temperature, humidity, weather)
  * ปัจจัยเชิงปฏิทิน (holiday, workingday)

---

## 4.2 **DataSet B**: Online Retail Dataset (Aggregation-based Feature Engineering)

[https://www.kaggle.com/datasets/carrie1/ecommerce-data](https://www.kaggle.com/datasets/carrie1/ecommerce-data)

### ลักษณะของข้อมูล

* ข้อมูลธุรกรรม e-commerce จริง
* หนึ่งแถว = หนึ่งรายการสินค้าในใบสั่งซื้อ

---

## 4.3 **Dataset C**: NYC Taxi Trip Duration (Interaction & Ratio Features)

[https://www.kaggle.com/competitions/nyc-taxi-trip-duration/data](https://www.kaggle.com/competitions/nyc-taxi-trip-duration/data)

### ลักษณะของข้อมูล

* ข้อมูลการเดินทางแท็กซี่ในนิวยอร์ก
* มีพิกัดต้นทาง–ปลายทาง และเวลาเดินทาง

---

In [ ]:
from google.colab import files
uploaded = files.upload()
uploaded.keys()

In [ ]:
import pandas as pd

housing_df = pd.read_csv('AmesHousing.csv')
A_df = pd.read_csv('bike_sharing.csv')
B_df = pd.read_csv('e_commerce_data.csv')
C_df = pd.read_csv('nyc-taxi.csv')

# 5) Time-based Feature Engineering

*(Temporal Representation)*

## แนวคิด

ข้อมูลจำนวนมากมี “เวลา” อยู่แล้ว
แต่ **timestamp ดิบไม่ใช่ feature ที่โมเดลเข้าใจโดยตรง**

Feature engineering ทำหน้าที่:

* แยกมิติของเวลาออกเป็นองค์ประกอบที่มีความหมาย
* แปลงลำดับเวลา → พฤติกรรม / ฤดูกาล / แนวโน้ม

---

## รูปแบบ Feature

* Calendar-based: hour, day, month, weekday
* Binary flags: is_weekend, is_holiday
* Temporal dependency: lag, rolling statistics

---

## ตัวอย่างจาก Dataset A (Bike Sharing)

In [ ]:
A_df['datetime'] = pd.to_datetime(A_df['datetime'])

A_df['hour'] = A_df['datetime'].dt.hour
A_df['dayofweek'] = A_df['datetime'].dt.dayofweek
A_df['is_weekend'] = A_df['dayofweek'].isin([5, 6]).astype(int)
A_df['month'] = A_df['datetime'].dt.month

A_df[['datetime', 'hour', 'dayofweek', 'is_weekend', 'month']].head()

* `hour` → routine รายวัน
* `is_weekend` → พฤติกรรมมนุษย์
* `month` → seasonality ระยะยาว

## Lag & Rolling Feature

In [ ]:
A_df['count_lag_1'] = A_df['count'].shift(1)
A_df['count_roll_24'] = A_df['count'].rolling(24).mean()

A_df[['count', 'count_lag_1', 'count_roll_24']].head(30)

* lag = อิทธิพลจากอดีตทันที
* rolling = แนวโน้มระยะสั้น
* ต้องระวัง **data leakage**

# 6) Aggregation-based Feature Engineering

*(Changing the Unit of Analysis)*

## แนวคิด

ข้อมูลเชิงธุรกรรม (transactional)
→ หนึ่งแถว ≠ หนึ่ง entity ที่โมเดลควรเรียนรู้

Feature engineering ทำหน้าที่:

* เปลี่ยน unit of analysis
* สรุปพฤติกรรมในระดับ entity (เช่น ลูกค้า)

---

## รูปแบบ Feature

* Sum / Mean / Count
* Frequency
* Temporal aggregation (recency-like)

---

## ตัวอย่างจาก Dataset B (Online Retail)

In [ ]:
B_df['InvoiceDate'] = pd.to_datetime(B_df['InvoiceDate'])
B_df['amount'] = B_df['Quantity'] * B_df['UnitPrice']

cust_features = (
  B_df.groupby('CustomerID')
    .agg(
      total_spent=('amount', 'sum'),
      avg_spent=('amount', 'mean'),
      num_invoices=('InvoiceNo', 'nunique'),
      total_items=('Quantity', 'sum')
    )
    .reset_index()
)

cust_features.head()

* aggregation ไม่ใช่แค่ “สรุป”
* แต่คือการออกแบบ representation ใหม่ของลูกค้า

## Temporal Aggregation (Recency-like)

In [ ]:
last_date = B_df['InvoiceDate'].max()

cust_features['recency_days'] = (
  B_df.groupby('CustomerID')['InvoiceDate']
    .max()
    .apply(lambda x: (last_date - x).days)
    .values
)

cust_features.head()

* feature นี้ไม่มีในข้อมูลดิบ
* แต่สะท้อนพฤติกรรมเชิงเวลาได้ชัด

# 7) Interaction & Ratio Feature Engineering

*(Encoding Relationships and Physical Meaning)*

## แนวคิด

ตัวแปรดิบหลายตัว “ไม่มีความหมายลำพัง”  
แต่มีความหมายเมื่อ **รวมกันอย่างมีเหตุผล**

Feature engineering ทำหน้าที่:

* encode ความสัมพันธ์
* สร้าง feature ที่สะท้อนกลไกจริงในโลก

---

## รูปแบบ Feature

* Ratio: x / y
* Difference: x − y
* Interaction: x × y

---

## ตัวอย่างจาก Dataset C (NYC Taxi)

### Distance Feature

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
  R = 6371
  dlat = radians(lat2 - lat1)
  dlon = radians(lon2 - lon1)
  a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
  return 2 * R * atan2(sqrt(a), sqrt(1-a))

C_df['distance_km'] = C_df.apply(
  lambda x: haversine(
    x['pickup_latitude'], x['pickup_longitude'],
    x['dropoff_latitude'], x['dropoff_longitude']
  ),
  axis=1
)

C_df[['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'distance_km']].head()

### Speed (Ratio Feature)

In [ ]:
C_df['speed_kmh'] = C_df['distance_km'] / (C_df['trip_duration'] / 3600)

C_df[['trip_duration', 'distance_km', 'speed_kmh']].head()

* lat/long → โมเดลไม่เข้าใจ
* speed → ตัวแทนสภาพจราจร (latent factor)

# 8) Distribution Transform

*(Skewness as Feature Engineering)*

## แนวคิด

ข้อมูลจริงมัก **skew**  
โมเดลเรียนรู้ distribution ที่ไม่สมดุลได้ยาก

การ transform:

* เปลี่ยนรูปการกระจาย
* เปลี่ยน “ความหมายเชิงสเกล” ของ feature

---

## Log / Log1p Transform

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

col = 'Lot Area'
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.hist(housing_df[col].dropna(), bins=30)
plt.title(f"Raw distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")

col_log = np.log1p(housing_df[col].dropna())

plt.subplot(1, 2, 2)
plt.hist(col_log, bins=30)
plt.title(f"LogTransform distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Power Transform

### Positive skewness

In [ ]:
from sklearn.preprocessing import PowerTransformer

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.hist(housing_df[col].dropna(), bins=30)
plt.title(f"Raw distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")

pt = PowerTransformer(method='yeo-johnson')
col_pt = pt.fit_transform(housing_df[[col]].dropna())

plt.subplot(1, 2, 2)
plt.hist(col_pt, bins=30)
plt.title(f"PowerTransform distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Negative skewness

In [ ]:
col = 'Year Built'

plt.figure(figsize=(15, 3))
plt.subplot(1, 3, 1)
plt.hist(housing_df[col].dropna(), bins=30)
plt.title(f"Raw distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")

col_log = np.log1p(housing_df[col].dropna())

plt.subplot(1, 3, 2)
plt.hist(col_log, bins=30)
plt.title(f"LogTransform distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")

pt = PowerTransformer(method='yeo-johnson')
col_pt = pt.fit_transform(housing_df[[col]].dropna())

plt.subplot(1, 3, 3)
plt.hist(col_pt, bins=30)
plt.title(f"PowerTransform distribution: {col}")
plt.xlabel(col)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# 9) Domain-driven Feature Engineering

*(Encoding Expert Knowledge into Data)*

## แนวคิดหลัก

**Domain-driven feature engineering** คือ  
การออกแบบ feature โดยใช้ *ความรู้เชิงบริบท* ที่ไม่ได้อยู่ในข้อมูลดิบ

> โมเดลไม่รู้จัก “ฤดูฝน”  
> โมเดลไม่รู้จัก “ชั่วโมงเร่งด่วน”  
> โมเดลไม่รู้จัก “พฤติกรรมมนุษย์”  
> → เราต้อง encode สิ่งเหล่านี้ให้เป็น feature

---

## ลักษณะสำคัญ

* ไม่ได้มาจากสูตร
* ไม่ได้มาจาก distribution
* มาจาก **ความเข้าใจข้อมูลในบริบทจริง**

---

## ตัวอย่างเชิงแนวคิด

| โดเมน       | ความรู้มนุษย์      | Feature ที่สร้าง    |
| ----------- | ------------------ | ------------------- |
| เวลา        | วันทำงาน ≠ วันหยุด | is_weekend          |
| สภาพอากาศ   | ฤดูมีผลต่อพฤติกรรม | season              |
| เมือง       | ชั่วโมงเร่งด่วน    | is_peak_hour        |
| สิ่งแวดล้อม | ค่าฝุ่นสูงผิดปกติ  | high_pollution_flag |

---

## ตัวอย่างจาก Dataset A (Bike Sharing)

In [ ]:
A_df['is_peak_hour'] = A_df['hour'].isin([7, 8, 17, 18]).astype(int)

In [ ]:
A_df[['hour', 'is_peak_hour']].sample(10)

* peak hour “ไม่ได้มาจากข้อมูล”
* แต่มาจากความเข้าใจชีวิตคนเมือง
* feature นี้ช่วยโมเดลแยก pattern ได้ดีกว่า hour อย่างเดียว

## ตัวอย่างจาก Dataset B (Online Retail)

In [ ]:
B_df['is_bulk_order'] = (B_df['Quantity'] >= 10).astype(int)

In [ ]:
B_df[['Quantity', 'is_bulk_order']].sample(10)

* คำว่า “bulk” ไม่มีอยู่ในข้อมูล
* แต่มีผลต่อพฤติกรรมและรายได้
* threshold มาจาก *business intuition* ไม่ใช่สูตร

## ตัวอย่างจาก Dataset C (NYC Taxi)

In [ ]:
C_df['pickup_hour'] = pd.to_datetime(C_df['pickup_datetime']).dt.hour
C_df['is_night_trip'] = (
  C_df['pickup_hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
)

In [ ]:
C_df[['pickup_datetime', 'pickup_hour', 'is_night_trip']].sample(10)

* การเดินทางกลางคืน ≠ กลางวัน
* encode ความเสี่ยง / traffic / demand ที่ต่างกัน

# 10) Statistical Feature Engineering

*(Summarizing Patterns and Deviations)*

## แนวคิดหลัก

Statistical features ไม่ได้พยายามอธิบายโลก
แต่พยายาม **สรุปพฤติกรรมของข้อมูลเอง**

> ถ้า domain-driven = ความรู้จากมนุษย์  
> statistical = ความรู้จากข้อมูล

---

## กลุ่มของ Statistical Features

### 1. Central Tendency

* mean, median

### 2. Dispersion

* std, variance

### 3. Relative Position

* z-score
* percentile

### 4. Local Statistics

* rolling mean
* rolling std

---

## ตัวอย่างจาก Dataset A (Bike Sharing)

In [ ]:
A_df['count_roll_mean_24'] = A_df['count'].rolling(24).mean()
A_df['count_roll_std_24'] = A_df['count'].rolling(24).std()

A_df[['count', 'count_roll_mean_24', 'count_roll_std_24']].iloc[45:55]

* mean = แนวโน้ม
* std = ความผันผวน
* ใช้คู่กันจึงเห็น “พฤติกรรม”

## ตัวอย่างจาก Dataset B (Online Retail)

In [ ]:
cust_features['spend_zscore'] = (
  cust_features['total_spent'] -
  cust_features['total_spent'].mean()
) / cust_features['total_spent'].std()

In [ ]:
cust_features[['total_spent', 'spend_zscore']].sample(10)

* z-score บอกว่า “ลูกค้านี้ผิดปกติแค่ไหน”
* เป็น feature เชิงเปรียบเทียบ ไม่ใช่ absolute

## ตัวอย่างจาก Dataset C (NYC Taxi)

In [ ]:
C_df['speed_z'] = (
    C_df['speed_kmh'] -
    C_df['speed_kmh'].mean()
) / C_df['speed_kmh'].std()

In [ ]:
C_df[['speed_kmh', 'speed_z']].sample(10)

## Domain-driven vs Statistical: เปรียบเทียบตรง ๆ

| ประเด็น    | Domain-driven  | Statistical                  |
| ---------- | -------------- | ---------------------------- |
| แหล่งที่มา | ความรู้มนุษย์  | รูปแบบในข้อมูล               |
| ความเข้าใจ | สูง            | ปานกลาง–ต่ำ                  |
| ความเสี่ยง | bias เชิงบริบท | sensitivity ต่อ distribution |
| การใช้ซ้ำ  | จำกัดโดเมน     | ใช้กว้างกว่า                 |

> **Feature ที่ดีมักเกิดจากการใช้ “สองแบบร่วมกัน”**

# 11) Hands-on: PM2.5 Dataset

ใช้ข้อมูล pm2.52021.csv - pm2.52024.csv ที่รวมกันเป็น time-series จาก lab ที่ผ่านมา

In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df_2021 = pd.read_csv("pm2.52021.csv")
df_2022 = pd.read_csv("pm2.52022.csv")
df_2023 = pd.read_csv("pm2.52023.csv")
df_2024 = pd.read_csv("pm2.52024.csv")

for df in [df_2021, df_2022, df_2023]:
    df["date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")

df_2024["date"] = pd.to_datetime(df_2024["Date"], format="%d/%m/%Y")

pm25_df = pd.concat(
    [df_2021, df_2022, df_2023, df_2024],
    ignore_index=True
).sort_values("date").reset_index(drop=True)

Saving pm2.52021.csv to pm2.52021.csv
Saving pm2.52022.csv to pm2.52022.csv
Saving pm2.52023.csv to pm2.52023.csv
Saving pm2.52024.csv to pm2.52024.csv


## Task 1: Time-based Features

* สร้าง `dayofweek`, `is_weekend`, `month`, `season`  
**ex.**
```python
df['season'] = df['month'].map({
    12:'Cool',1:'Cool',2:'Cool',
    3:'Hot',4:'Hot',5:'Hot',
    6:'Rainy',7:'Rainy',8:'Rainy',9:'Rainy',10:'Rainy',11:'Cool'
})
```
* อภิปราย: feature ใดสะท้อนพฤติกรรมมนุษย์มากที่สุด

In [4]:
# สร้าง time-based features
pm25_df["dayofweek"] = pm25_df["date"].dt.day_name()
pm25_df["is_weekend"] = pm25_df["date"].dt.dayofweek.isin([5, 6]).astype(int)
pm25_df["month"] = pm25_df["date"].dt.month

pm25_df["season"] = pm25_df["month"].map({
    12:"Cool", 1:"Cool", 2:"Cool",
    3:"Hot",   4:"Hot",  5:"Hot",
    6:"Rainy", 7:"Rainy", 8:"Rainy", 9:"Rainy", 10:"Rainy", 11:"Cool"
})

print(pm25_df[["date","dayofweek","is_weekend","month","season"]].head())
print(pm25_df[["is_weekend","month"]].describe())

        date dayofweek  is_weekend  month season
0 2021-01-01    Friday           0      1   Cool
1 2021-01-02  Saturday           1      1   Cool
2 2021-01-03    Sunday           1      1   Cool
3 2021-01-04    Monday           0      1   Cool
4 2021-01-05   Tuesday           0      1   Cool
        is_weekend        month
count  1277.000000  1277.000000
mean      0.285043     6.094753
std       0.451612     3.426059
min       0.000000     1.000000
25%       0.000000     3.000000
50%       0.000000     6.000000
75%       1.000000     9.000000
max       1.000000    12.000000


is_weekend สะท้อนพฤติกรรมมนุษย์มากที่สุด เนื่องจากวันปกติและวันหยุดมนุษย์จะมีพฤติกรรมที่แตกต่างกัน เช่น วันปกติต้องไปทำงาน วันหยุดพักผ่อนอยู่ที่บ้าน

## Task 2: Lag & Rolling Features

* สร้างค่า pm2.5 lag 1 วัน
* สร้างค่า pm2.5 rolling 7 วัน
* เปรียบเทียบ lag vs rolling
* อธิบายความหมายเชิงเวลา

In [7]:
station_cols = [col for col in pm25_df.columns if col.endswith("T")]

pm25_df["pm25_mean"] = pm25_df[station_cols].mean(axis=1)

# สร้างค่า pm2.5 lag 1 วัน
pm25_df["pm25_lag1"] = pm25_df["pm25_mean"].shift(1)

# สร้างค่า pm2.5 rolling 7 วัน
pm25_df["pm25_rolling7"] = pm25_df["pm25_mean"].rolling(window=7).mean()

print(pm25_df[["date", "pm25_mean", "pm25_lag1", "pm25_rolling7"]].head(15))

         date  pm25_mean  pm25_lag1  pm25_rolling7
0  2021-01-01  21.661972        NaN            NaN
1  2021-01-02  25.916667  21.661972            NaN
2  2021-01-03  34.180556  25.916667            NaN
3  2021-01-04  30.902778  34.180556            NaN
4  2021-01-05  33.361111  30.902778            NaN
5  2021-01-06  33.736111  33.361111            NaN
6  2021-01-07  30.666667  33.736111      30.060837
7  2021-01-08  22.944444  30.666667      30.244048
8  2021-01-09  20.041667  22.944444      29.404762
9  2021-01-10  26.250000  20.041667      28.271825
10 2021-01-11  27.416667  26.250000      27.773810
11 2021-01-12  25.549296  27.416667      26.657836
12 2021-01-13  39.126761  25.549296      27.427929
13 2021-01-14  55.042254  39.126761      30.910155
14 2021-01-15  61.416667  55.042254      36.406187


pm25_lag1 (Lag 1 วัน)
คือ การบอกว่า เมื่อวานค่า PM2.5 เป็นเท่าไหร่ โดยถ้าอากาศมีมลพิษที่สะสมอยู่ในอากาศเมื่อวาน มลพิษนั้นก็จะยังไม่ได้หายไปทันที แต่ยังคงส่งผลต่อวันถัดไปด้วย เช่น ถ้าเมื่อวาน PM2.5 สูงเพราะไฟป่า วันนี้ก็มีแนวโน้มสูงตามด้วย

pm25_rolling7 (Rolling Mean 7 วัน)
คือ ค่าเฉลี่ยของ 7 วันที่ผ่านมา บ่งบอกถึง trend ของสัปดาห์นั้นๆ ช่วยตัดสัญญาณรบกวนรายวันออก เช่น วันที่ฝนตกหนักค่าอาจดิ่งลงผิดปกติ แต่ rolling จะยังสะท้อนภาพรวมของสัปดาห์ได้ดีกว่า

## Task 3: Aggregation by Group

* เฉลี่ย PM2.5 รายเดือน / รายฤดูกาล
* อธิบายว่า aggregation ทำให้ข้อมูล “สูญเสียอะไร” บ้าง

In [10]:
# รายเดือน

monthly_avg = pm25_df.groupby("month")["pm25_mean"].mean()
monthly_avg = monthly_avg.reset_index()
monthly_avg.columns = ["month", "pm25_monthly_avg"]
print("รายเดือน:")
print(monthly_avg)

# รายฤดูกาล

seasonal_avg = pm25_df.groupby("season")["pm25_mean"].mean()
seasonal_avg = seasonal_avg.reset_index()
seasonal_avg.columns = ["season", "pm25_seasonal_avg"]
print("\nรายฤดูกาล:")
print(seasonal_avg)

รายเดือน:
    month  pm25_monthly_avg
0       1         32.939860
1       2         37.074293
2       3         38.633625
3       4         35.441576
4       5         18.881040
5       6         12.501004
6       7         11.270769
7       8         11.522857
8       9         12.139036
9      10         16.873498
10     11         20.129081
11     12         27.164772

รายฤดูกาล:
  season  pm25_seasonal_avg
0   Cool          30.028283
1    Hot          30.936977
2  Rainy          12.845964


สิ่งที่สูญเสีย
1. Variance เพราะ เดือนที่มีบางวันที่ PM2.5 พุ่งสูงผิดปกติ แต่บางวันปกติ เมื่อนำทั้งเดือนมาเฉลี่ยกันความแปรปรวนจะถูกหักล้าง
2. Temporal order หายไป เพราะ ไม่รู้ว่าค่าที่สูงอยู่ต้นเดือนหรือสิ้นเดือน
3. Outlier ถูกกลืน เพราะ เหตุการณ์พิเศษ เช่น ไฟป่าลุกลาม ถูกเฉลี่ยรวมกับค่าฝุ่นของวันปกติจนค่าผิดปกติหายไป

## Task 4: บันทึกเป็น **Feature-Enhanced PM2.5 Dataset v1.csv**

In [11]:
pm25_df.to_csv("Feature-Enhanced PM2.5 Dataset v1.csv", index=False)

## Task 5: Interpretation (ไม่มีคำตอบตายตัว)

* Feature ใด **ไม่ควรใช้** หากต้องพยากรณ์อนาคต? เพราะอะไร

---

pm25_mean ของวันที่ต้องการพยากรณ์ เพราะ target ถูกนำมาใส่เป็น feature จะทำให้เกิด Data Leakage

Feature ที่ใช้ได้ คือ pm25_lag1 และ pm25_rolling7 เพราะ เป็นการอ้างอิงข้อมูลอดีตที่รู้แล้ว รวมถึง dayofweek, is_weekend, month, season ที่รู้ล่วงหน้าได้

# 12) Deliverable Checklist

* [ ] Notebook นี้ที่ทำ Hands-on แล้ว
* [ ] Feature-Enhanced PM2.5 Dataset v1.csv